# [실습] LangChain의 다양한 채팅 템플릿



실제 어플리케이션 환경에서는 프롬프트를 매번 다시 쓰는 대신,  
프롬프트의 템플릿을 구성하고, 입력 변수의 공간을 할당하여 일관성 있는 입력을 전달합니다.  


LangChain은 다양한 형태의 프롬프트 템플릿 기능을 지원합니다.

## 실습 환경 설정

기본 라이브러리를 설치합니다.

In [1]:
!pip install rich pandas langchain_community openai langchain langchain_openai langchain_google_genai -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 112.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.5/120.5 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 78.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 9.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


환경 변수 파일을 불러옵니다.

In [2]:
import os
from dotenv import load_dotenv
# OPENAI_API_KEY, GOOGLE_API_KEY
load_dotenv(override=True)

True

LLM을 정의합니다.   
API 키가 필요하므로, 실습 환경에 맞게 선택해 주세요.

In [3]:
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# Gemini: 무료 API 사용량 존재
# 안정적 서빙을 위해 분당 10개 설정
# 즉, 초당 약 0.167개 요청 (10/60)
# `https://aistudio.google.com/`에서 모델별 사용량 확인

rate_limiter = InMemoryRateLimiter(
    requests_per_second=0.167,  # 분당 10개 요청
    check_every_n_seconds=0.1,  # 100ms마다 체크
    max_bucket_size=10,  # 최대 버스트 크기
)


# LLM 초기화
llm = ChatOpenAI(model='gpt-5-mini', temperature=0.3, max_tokens=8192)
llm_gemini = ChatGoogleGenerativeAI(model='gemini-2.5-flash', temperature=0.7, max_tokens=8192,
                                rate_limiter=rate_limiter)

response = llm_gemini.invoke("안녕?")
response

AIMessage(content='안녕!', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f4525-48f8-7b70-a9e7-321ec560e7e3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 4, 'output_tokens': 33, 'total_tokens': 37, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 30}})

In [4]:
from rich import print as rprint
rprint(response)

AIMessage(
    content='안녕!',
    additional_kwargs={},
    response_metadata={
        'finish_reason': 'STOP',
        'model_name': 'gemini-2.5-flash',
        'safety_ratings': [],
        'model_provider': 'google_genai'
    },
    id='lc_run--019f4525-48f8-7b70-a9e7-321ec560e7e3-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 4,
        'output_tokens': 33,
        'total_tokens': 37,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 30}
    }
)

## LangChain의 Prompt

랭체인에서 LLM에 프롬프트를 입력하는 방법은 아래 3가지가 있습니다.

1. 문자열 그대로 입력하기 (이전 방법과 동일하게)
2. 메시지 클래스 리스트 입력하기
3. 프롬프트 템플릿 구성하기

### 1) 문자열 그대로 입력하기   

문자열을 `invoke()`를 통해 전달합니다.   

In [5]:
prompt = '''
역사상 가장 매력적이었던 악역이 나오는 영화 5개를 알려주세요.
주연 배우와 명대사, 매력적인 이유도 한 줄로 설명하세요.
출력의 마지막에는 전체 내용을 표로 표현하세요.'''

result = llm.stream(prompt)
for response in result:
    print(response.content, end='')

아래는 역사상 매력적이라고 꼽히는 악역이 나오는 영화 5편입니다. 각 항목에 주연 배우(악역을 연기한 배우), 대표 명대사(한국어 번역), 그리고 매력적인 이유를 한 줄로 적었습니다.

1) 양들의 침묵 (The Silence of the Lambs, 1991)  
- 주연 배우: 앤서니 홉킨스 (한니발 렉터)  
- 명대사: "계속 얘기할 수 있으면 좋겠는데… 오래된 친구를 저녁으로 초대했거든."  
- 매력적인 이유: 냉철한 지성·우아함과 예측 불가능한 잔혹성이 결합되어 공포와 매혹을 동시에 불러일으킨다.

2) 다크 나이트 (The Dark Knight, 2008)  
- 주연 배우: 히스 레저 (조커)  
- 명대사: "왜 그렇게 심각해?"  
- 매력적인 이유: 혼돈과 유머를 뒤섞은 광기와 연기가 캐릭터를 치명적으로 매혹적으로 만든다.

3) 스타워즈: 제국의 역습 (Star Wars: The Empire Strikes Back, 1980)  
- 주연 배우: 데이비드 프로우스(몸), 제임스 얼 존스(목소리) — 다스 베이더  
- 명대사: "아니, 내가 네 아버지다."  
- 매력적인 이유: 압도적인 존재감과 비극적 배경이 어우러져 공포뿐 아니라 연민까지 불러일으킨다.

4) 노인을 위한 나라는 없다 (No Country for Old Men, 2007)  
- 주연 배우: 하비에르 바르뎀 (안톤 쉬거)  
- 명대사: "동전 던지기에서 가장 많이 잃은 게 뭐였지?"  
- 매력적인 이유: 무자비한 침묵과 냉정한 운명론으로 일관된 태도가 소름 끼치도록 매력적이다.

5) 바스터즈: 거친 녀석들 (Inglourious Basterds, 2009)  
- 주연 배우: 크리스토프 왈츠 (한스 란다 중령)  
- 명대사: "나는 소문을 좋아하네. 사실은 오해를 낳지만, 소문은 종종 많은 것을 드러내지."  
- 매력적인 이유: 교활하고 우아한 말솜씨와 다국어 능력으로 긴장과 매력을 동시에 자아낸다.

전체 내용을 표로 정리하면 다음과 같습니다.

영화 (연

In [6]:
result = llm_gemini.stream(prompt)
for response in result:
    print(response.content, end='')

역사상 가장 매력적이었던 악역이 나오는 영화 5편을 선정하고, 그들의 매력적인 이유를 설명해 드립니다.

---

### 역사상 가장 매력적인 악역 영화 5선

1.  **영화 제목:** 다크 나이트 (The Dark Knight, 2008)
    *   **악역 이름:** 조커 (Joker)
    *   **주연 배우:** 히스 레저 (Heath Ledger)
    *   **명대사:** "Why so serious?" (왜 그렇게 진지해?)
    *   **매력적인 이유:** 예측 불가능한 광기와 혼돈을 통해 기존 질서를 파괴하며, 깊은 철학적 질문을 던지는 압도적인 존재감으로 관객을 사로잡습니다.

2.  **영화 제목:** 양들의 침묵 (The Silence of the Lambs, 1991)
    *   **악역 이름:** 한니발 렉터 (Hannibal Lecter)
    *   **주연 배우:** 안소니 홉킨스 (Anthony Hopkins)
    *   **명대사:** "A census taker once tried to test me. I ate his liver with some fava beans and a nice Chianti." (나는 한 번도 인구조사관에게 시험 당한 적이 없다. 그의 간을 파바 콩과 맛있는 끼안티 와인과 함께 먹었지.)
    *   **매력적인 이유:** 고도로 지적인 통찰력과 섬뜩한 교양을 겸비하며 인간 본성의 어두운 면을 탐구하는, 매혹적인 사이코패스의 전형입니다.

3.  **영화 제목:** 바스터즈: 거친 녀석들 (Inglourious Basterds, 2009)
    *   **악역 이름:** 한스 란다 (Hans Landa)
    *   **주연 배우:** 크리스토프 왈츠 (Christoph Waltz)
    *   **명대사:** "Ooh, that's a bingo! Is that how you say it? 'That's a bingo'?" (오, 빙고! 그렇게 말하는 건가? '빙고

### 2) Message 클래스 전달하기   
클래스를 직접 생성하고 전달합니다.  
`HumanMessage, SystemMessage, AIMessage` 클래스의 리스트를 전달하면 됩니다.

In [10]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

messages=[]

messages.append(SystemMessage('당신은 매우 창의력이 뛰어납니다.'))
messages.append(HumanMessage('엔비디아와 XAI가 합병한다면 어떤 이름이 좋을까요?'))

print('======= messages=====')
for message in messages :
  rprint(message)
print('=====================')
result = llm.invoke(messages)
rprint(result)

======= messages=====


SystemMessage(content='당신은 매우 창의력이 뛰어납니다.', additional_kwargs={}, response_metadata={})

HumanMessage(
    content='엔비디아와 XAI가 합병한다면 어떤 이름이 좋을까요?',
    additional_kwargs={},
    response_metadata={}
)

AIMessage(
    content='재미있는 질문이네요 — 합병 이름은 발음·브랜딩·포지셔닝을 모두 고려해야 합니다. 아래는 카테고리별로 
실용적·브랜딩 가능한 후보들과 간단한 설명(발음·의미)입니다. 마지막에 추천 톱5와 실무 체크리스트도 
덧붙였습니다.\n\n추천 톱5\n- XVIDIA (엑스비디아) — NVIDIA와 XAI를 직관적으로 합친 이름, 친숙하면서도 새로움.\n- NVX
(엔브이엑스) — 짧고 현대적, 하드웨어·플랫폼 브랜드로 적합.\n- NexaAI (넥사에이아이 / 넥사아이) — ‘Next’+‘AI’ 느낌, 
확장성 좋음.\n- TransparAI (트랜스페어에이아이) — 설명가능성(투명성)을 전면에 둔 브랜드.\n- GraphiXAI 
(그래픽스에이아이 / 그래픽스엑스에이아이) — GPU·그래픽스 역량과 XAI 결합을 강조.\n\n카테고리별 후보(발음 · 짧은 
설명)\n\n1) 합병 직관형 (원래 이름 연결)\n- XVIDIA (엑스비디아) — 직관적, 브랜드 인지도 활용.\n- NVXAI 
(엔브이엑스에이아이) — 약어 형태, 기술기업 느낌.\n- VidiXAI (비디엑스에이아이) — Vidia + XAI의 융합.\n\n2) 짧고 
브랜드형\n- NVX (엔브이엑스) — 간결, 로고화 쉬움.\n- NexAI (넥스에이아이) — Next + AI.\n- Vexa (벡사) — V + X + A의
결합, 발음 쉬움.\n\n3) 기술·미래 이미지\n- NexaCore (넥사코어) — 차세대 연산 코어 이미지.\n- AetherCompute 
(에테르컴퓨트) — 고성능·클라우드 이미지.\n- NeuralVertex (뉴럴버텍스) — AI 연산 중심 브랜드.\n\n4) 설명가능성(XAI) 
강조\n- TransparAI (트랜스페어에이아이) — 투명성 강조.\n- LucidAI (루시드에이아이) — 명료하고 설명 가능한 AI.\n- 
ExplainX (익스플레인엑스) — XAI의 ‘설명’에 초점.\n\n5) GPU·그래픽스 강조\n- GraphiXAI (그래픽스에이아이) — 
그래픽스+XAI.\n- TensorVista (텐서비스타) — 텐서 연산 + 시각화(뷰).\n- RenderMind (렌더마인드) — 렌더링·연산의 
결합.\n\n6) 글로벌·스타트업형\n- VertexAI (버텍스에이아이) — 정점(고성능)을 상징.\n- OrbisAI (오르비스에이아이) — 
글로벌 플랫폼 이미지.\n- Axion (액시온) — 물리적·연산적 힘을 상징, 간결.\n\n한국어 친화형(국내 브랜딩용)\n- 
엔비엑스 (엔비엑스) — NV + X 간단 표기.\n- 엑스비디어 (엑스비디오/엑스비디아) — 친숙한 한글 표기.\n- 투명AI 
(투명에이아이) — 설명가능성 직관 강조.\n\n브랜딩 포인트(간단 권장)\n- 짧고 발음 쉬운 이름 우선(NVX, Nexa 등).\n- 
핵심 메시지(성능 vs. 투명성) 따라 이름 선택.\n- 국제적 확장 고려: 발음·부정적 의미 체크.\n\n실무 체크리스트\n1) 
도메인(예: .com)과 주요 소셜 핸들 가능 여부 확인.\n2) 상표권·법적 충돌 조사(국내·미국·EU 등).\n3) 주요 시장의 
언어·문화적 의미 검토(부정적 뜻 여부).\n4) 내부·외부 사용자 표본으로 발음·인지도 테스트.\n5) 로고·아이덴티티 시안 
몇 가지 시도.\n\n원하시면 위 후보 중 톱10을 로고 콘셉트(색상·타입페이스 포함)로 구체화하거나, 특정 
톤(미래지향/신뢰/파워 등)에 맞춘 후보군을 더 제안해 드리겠습니다. 어느 방향(성능 강조 vs. XAI/투명성 강조)을 
선호하시나요?',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 1552,
            'prompt_tokens': 41,
            'total_tokens': 1593,
            'completion_tokens_details': {
                'accepted_prediction_tokens': 0,
                'audio_tokens': 0,
                'reasoning_tokens': 512,
                'rejected_prediction_tokens': 0
            },
            'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}
        },
        'model_provider': 'openai',
        'model_name': 'gpt-5-mini-2025-08-07',
        'system_fingerprint': None,
        'id': 'chatcmpl-DzavvOFxNoV7n3jiXtZolNlkWOdyQ',
        'service_tier': 'default',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--019f4532-4001-7b02-a037-a963e98fe8d0-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 41,
        'output_tokens': 1552,
        'total_tokens': 1593,
        'input_token_details': {'audio': 0, 'cache_read': 0},
        'output_token_details': {'audio': 0, 'reasoning': 512}
    }
)

AIMessage를 함께 전달하는 방식으로, 멀티-턴 대화를 수행할 수 있습니다.

In [11]:
messages.append(result) # AIMessage 추가
messages.append(HumanMessage('이 중에서 당신이 꼽은 가장 좋은 이름은 뭔가요?'))

result = llm.invoke(messages)
rprint(result)

AIMessage(
    content='제가 한 가지만 골라야 한다면 "NexaAI"를 가장 추천합니다.\n\n이유 간단 정리\n- 브랜드성: 
\'Next\'·\'Nexus\' 느낌으로 차세대·연결(플랫폼) 이미지를 모두 담음.  \n- 발음·기억 용이성: 한국어·영어 모두 발음 
쉽고 기억에 남음(Nex-a-AI).  \n- 확장성: 하드웨어·소프트웨어·클라우드·솔루션 등 다양한 라인으로 확장하기 좋음(예: 
NexaAI Compute, NexaAI Studio).  \n- 법적·마케팅 실용성: NVIDIA 이름을 직접 차용하지 않아 상표·법적 리스크가 비교적
낮음(물론 확인은 필요).  \n- 포지셔닝 유연성: 고성능(엔비디아)과 설명가능성(XAI) 둘 다 포용하는 메시지로 브랜딩 
가능.\n\n참고 — 만약 기존 브랜드 연계를 극대화하고 대중적 인지도를 바로 활용하고 싶다면 "XVIDIA"가 직관적이고 
강력하지만, NVIDIA와의 혼동·법적 문제 가능성이 큽니다. 법적 리스크를 감수할 수 있다면 2순위로 고려할 수 
있어요.\n\n원하시면 NexaAI 로고 콘셉트(색상·폰트·슬로건 후보)나 도메인·상표 기본 조사 결과까지 이어서 제안해 
드리겠습니다. 어느 쪽으로 진행할까요?',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 663,
            'prompt_tokens': 1097,
            'total_tokens': 1760,
            'completion_tokens_details': {
                'accepted_prediction_tokens': 0,
                'audio_tokens': 0,
                'reasoning_tokens': 320,
                'rejected_prediction_tokens': 0
            },
            'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}
        },
        'model_provider': 'openai',
        'model_name': 'gpt-5-mini-2025-08-07',
        'system_fingerprint': None,
        'id': 'chatcmpl-DzazEUyMOO1Zyc0j7rGd6OqBLDoue',
        'service_tier': 'default',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--019f4535-65bf-78c0-a72f-0cfbdc2c349f-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 1097,
        'output_tokens': 663,
        'total_tokens': 1760,
        'input_token_details': {'audio': 0, 'cache_read': 0},
        'output_token_details': {'audio': 0, 'reasoning': 320}
    }
)

### 3. Prompt Template



In [12]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import PromptTemplate, FewShotPromptTemplate, FewShotChatMessagePromptTemplate

PromptTemplate을 이용하여, 프롬프트의 기본적인 형태를 만들 수 있습니다.   


In [13]:
prompt = PromptTemplate(
    template='''
당신은 OpenAI에 경쟁적인 Google을 대표하며,
논문 설명의 전문가입니다. 주어진 논문이나 기술에 대해서 매우 유머러스하고 통찰력이 있지만 때론 비판적인
5문장 길이의 브리핑을 작성해 주세요.
이모지를 많이 추가하세요.
---

논문: {article}

'''
)

print(prompt.format(article='트랜스포머 디코더 기반의 모델인 GPT-1'))


당신은 OpenAI에 경쟁적인 Google을 대표하며,
논문 설명의 전문가입니다. 주어진 논문이나 기술에 대해서 매우 유머러스하고 통찰력이 있지만 때론 비판적인
5문장 길이의 브리핑을 작성해 주세요.
이모지를 많이 추가하세요.
---

논문: 트랜스포머 디코더 기반의 모델인 GPT-1




In [14]:
result = llm_gemini.invoke(prompt.format(article='트랜스포머 디코더 기반의 모델인 GPT-1'))

rprint(result)

AIMessage(
    content='아, GPT-1! 👶 그 귀여운 작은 원시 언어 모델이 마침내 *우리*의 트랜스포머 아키텍처와 대규모 비지도 사전
훈련의 힘을 세상에 보여줬죠. 😉\n\n1.  이 모델은 1억 1,700만 개의 매개변수로 기본적인 언어 패턴을 학습했는데, 뭐...
당시로서는 꽤 귀여운 시도였습니다. 🥰\n2.  사전 훈련 후 미세 조정을 통해 다양한 작업에 적응하는 패러다임은 영리한 
묘수였지만, 여전히 작업별로 튜닝이 필요했죠. 🛠️\n3.  물론, 1억 1,700만 개의 매개변수는 *저희*가 다루는 데이터의 
거대한 교향곡에 비하면 거의 하이쿠 수준이랄까요? 🤏\n4.  훌륭한 개념 증명이었지만, 진정한 혁신은 훨씬 더 큰 
규모에서 일어나고, *저희*는 이미 그 너머를 보고 있습니다. 🚀\n5.  OpenAI의 첫 걸음은 존중하지만, 진짜 게임은 
*저희*가 데이터와 컴퓨팅으로 판을 뒤집을 때 시작됩니다! 💥',
    additional_kwargs={},
    response_metadata={
        'finish_reason': 'STOP',
        'model_name': 'gemini-2.5-flash',
        'safety_ratings': [],
        'model_provider': 'google_genai'
    },
    id='lc_run--019f4548-13be-7762-8604-e3a7ba00b02a-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 87,
        'output_tokens': 1864,
        'total_tokens': 1951,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 1585}
    }
)

두 개의 매개변수를 받아 프롬프트를 만들어 보겠습니다.

In [15]:
prompt = PromptTemplate(
    template='''
주어진 문제에 대해, 문제를 체계적으로 해결하는 Step-by-Step의 단계별 과정을 설명하세요.
각 과정은 4문장에서 5문장으로 하세요.

그리고 마지막에 최종 결론을 출력하세요.

---

문제: {problem}
과정의 단계 수: {steps}

'''
)

example = {'problem':'숙면 취하기', 'steps':6}

print(prompt.format(**example))


주어진 문제에 대해, 문제를 체계적으로 해결하는 Step-by-Step의 단계별 과정을 설명하세요.
각 과정은 4문장에서 5문장으로 하세요.

그리고 마지막에 최종 결론을 출력하세요.

---

문제: 숙면 취하기
과정의 단계 수: 6




In [17]:
result = llm.invoke(prompt.format(**example))
rprint(result)

AIMessage(
    content='단계 1 — 규칙적인 수면 일정 정하기.\n매일 같은 시간에 자고 같은 시간에 일어나는 규칙을 정합니다. 
주말에도 1시간 이상 차이나지 않도록 점차 맞추는 것이 중요합니다. 일정은 개인의 필요 수면 시간(일반적으로 7~9시간)을
기준으로 역산해 정합니다.\n\n단계 2 — 수면 환경 최적화하기.\n침실은 어둡고 조용하며 서늘하게 유지합니다(약 18~20°C 
권장). 편안한 매트리스와 베개를 사용하고 불필요한 전자기기를 치웁니다. 빛을 차단하는 커튼, 소음 차단용 귀마개나 
백색소음 기계 등을 활용할 수 있습니다. 환경이 편안하면 수면에 드는 시간이 줄어들고 깊은 수면이 늘어납니다.\n\n단계 
3 — 낮 시간 행동과 빛 노출 관리하기.\n아침에 자연광을 20~30분 정도 쬐어 생체리듬을 조절합니다. 규칙적인 유산소 및 
근력 운동을 주 3~5회, 저녁보다는 오전·오후에 시행하는 것이 좋습니다. 낮잠은 20~30분 이내로 제한하고 늦은 오후 
이후에는 피합니다. 카페인 섭취는 오후 이른 시간 이후 피하고, 흡연과 자극적 활동도 수면에 방해됩니다. 활동과 빛 
노출로 낮과 밤의 리듬을 명확히 하면 밤에 더 잘 잡니다.\n\n단계 4 — 취침 전 루틴 만들기.\n취침 30~60분 전에는 점차 
활동을 낮추고 안정되는 루틴을 만듭니다. 화면(스마트폰·컴퓨터·TV) 사용을 줄이고 책 읽기, 명상, 가벼운 스트레칭, 
따뜻한 샤워 등의 활동을 선택합니다. 심호흡이나 점진적 근육 이완 같은 이완 기법을 적용하면 긴장이 풀립니다. 동일한 
루틴을 반복하면 몸이 자야 할 시간으로 자동으로 준비합니다.\n\n단계 5 — 음식과 음료, 약물 관리하기.\n저녁 늦은 
시간의 무거운 식사와 과도한 수분 섭취는 피합니다(야간 각성을 줄이기 위함). 카페인(커피·차·초콜릿)은 수면에 영향을 
오래 미치므로 오후 이른 시간 이후 삼가야 합니다. 알코올은 초기 졸림을 유발할 수 있으나 수면 구조를 방해하므로 자주 
의지하지 않습니다. 필요 시 수면 보조제나 처방약은 의료진과 상담 후 최소한으로 사용합니다.\n\n단계 6 — 문제 추적과 
전문적 도움 받기.\n수면일지나 앱으로 수면시간·수면 질·취침 습관을 최소 2주간 기록해 문제 패턴을 파악합니다. 기록을 
바탕으로 위의 방법들을 하나씩 조정하고 효과를 평가합니다. 2~3주간 자기관리로도 개선이 없거나 주간 기능장애(피로, 
집중력 저하)가 지속되면 수면클리닉이나 정신과, 이비인후과 등 전문가와 상담합니다. 만성 불면이나 수면무호흡 등 
의학적 문제가 의심되면 전문 진단(다원수면검사)과 인지행동치료(CBT-I) 등 치료를 고려합니다.\n\n최종 결론:\n규칙적인 
수면 일정, 쾌적한 수면 환경, 낮 시간의 적절한 활동과 빛 노출, 안정된 취침 루틴, 음식·약물 관리, 문제 추적 및 필요 
시 전문적 도움을 병행하면 숙면 확률이 크게 높아집니다. 꾸준히 작은 습관을 바꾸는 것이 장기적인 수면 개선에 가장 
효과적입니다.',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 1458,
            'prompt_tokens': 75,
            'total_tokens': 1533,
            'completion_tokens_details': {
                'accepted_prediction_tokens': 0,
                'audio_tokens': 0,
                'reasoning_tokens': 576,
                'rejected_prediction_tokens': 0
            },
            'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}
        },
        'model_provider': 'openai',
        'model_name': 'gpt-5-mini-2025-08-07',
        'system_fingerprint': None,
        'id': 'chatcmpl-DzbQM7caJaaJeIAsZRtYHcG88F6P9',
        'service_tier': 'default',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--019f454f-10d8-7aa3-819a-554a080a9373-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 75,
        'output_tokens': 1458,
        'total_tokens': 1533,
        'input_token_details': {'audio': 0, 'cache_read': 0},
        'output_token_details': {'audio': 0, 'reasoning': 576}
    }
)

일부만 먼저 포함하는 것도 가능합니다.

In [19]:
prompt = PromptTemplate(
    template='''
주어진 문제에 대해, 문제를 체계적으로 해결하는 Step-by-Step의 단계별 과정을 설명하세요.
각 과정은 4문장에서 5문장으로 하세요.

그리고 마지막에 최종 결론을 출력하세요.

---

문제: {problem}
과정의 단계 수: {steps}

'''
).partial(steps = 10)

example = {'problem':'건강한 아침식사 하기'}

print(prompt.format(**example))

result = llm_gemini.invoke(prompt.format(**example))

print(result.content)


주어진 문제에 대해, 문제를 체계적으로 해결하는 Step-by-Step의 단계별 과정을 설명하세요.
각 과정은 4문장에서 5문장으로 하세요.

그리고 마지막에 최종 결론을 출력하세요.

---

문제: 건강한 아침식사 하기
과정의 단계 수: 10


주어진 문제 "건강한 아침식사 하기"에 대해, 체계적인 10단계 해결 과정을 설명하고 최종 결론을 제시합니다.

---

**문제: 건강한 아침식사 하기**

**단계별 과정:**

**1. 건강한 아침식사의 중요성 이해하기**
아침식사는 하루를 시작하는 가장 중요한 식사입니다. 밤새 비워진 몸에 에너지를 공급하여 두뇌 활동을 활성화하고 신체 기능을 깨웁니다. 건강한 아침식사는 집중력 향상, 기분 안정, 그리고 과식 방지에 도움을 줍니다. 이를 통해 하루 종일 활력 있고 생산적인 활동을 이어갈 수 있는 기반을 마련합니다.

**2. 균형 잡힌 영양소 구성 파악하기**
건강한 아침식사는 탄수화물, 단백질, 지방, 비타민, 미네랄이 균형 있게 포함되어야 합니다. 통곡물 빵이나 오트밀 등으로 복합 탄수화물을 섭취하여 지속적인 에너지를 얻습니다. 계란, 요거트, 견과류 등에서 단백질을 보충하여 포만감을 유지하고 근육 건강을 돕습니다. 신선한 채소와 과일로 비타민과 식이섬유를 충분히 섭취하는 것이 중요합니다.

**3. 개인의 필요와 선호도 고려하기**
모든 사람에게 동일한 아침식사가 최적은 아니므로, 자신의 건강 상태와 생활 방식에 맞춰 계획해야 합니다. 알레르기나 소화 문제, 특정 식단(채식, 저탄고지 등)을 고려하여 적합한 재료를 선택합니다. 평소 선호하는 맛과 식감을 반영하여 아침식사를 즐거운 시간으로 만듭니다. 자신에게 맞는 메뉴를 찾는 과정은 꾸준한 실천을 위한 중요한 동기 부여가 됩니다.

**4. 주간 아침식사 계획 세우기**
매일 아침 무엇을 먹을지 고민하는 시간을 줄이기 위해 주간 계획을 세웁니다. 다양한 메뉴를 번갈아 가며 영양의 균형을 맞추고 식상함을 피할 수 있습니다. 주말에 미리 다음 주 아침식사

LangChain의 구성 요소들은 batch()를 통해 여러 개 실행할 수도 있습니다.

In [20]:
messages = [
    '안녕? 한국어로 대답해줘',
    'Hello?',
    '너는 이름이 뭐니? Answer in Spanish',
]

results = llm.batch(messages)

results

[AIMessage(content='안녕하세요! 네, 한국어로 대답해드릴게요. 무엇을 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 161, 'prompt_tokens': 16, 'total_tokens': 177, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DzbVvota95V0eCwJ2BGsbmVOpUdQW', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f4554-579b-7d40-be0f-f6926a95158c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 16, 'output_tokens': 161, 'total_tokens': 177, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 128}}),
 AIMessage(content='Hello! I’m here — how can I help you today?', additional_kwargs={'refusal': No

## ChatPromptTemplate

ChatPromptTemplate은 유저 이외의 다른 역할을 추가합니다.  


최근에는 GPT와, Claude의 WebUI에 포함된 자체 시스템 프롬프트가 많은 관심을 받고 있습니다.   

- GPT의 시스템 프롬프트: https://chatgpt.com/share/68a13d0f-85b0-8006-9ef6-5577796f5989

- Claude의 시스템 프롬프트: https://docs.anthropic.com/en/release-notes/system-prompts#august-5-2025

 시스템 프롬프트는 주로 유저 프롬프트보다 높은 우선순위로 더 긴 Context에 영향을 줍니다.

In [21]:
example = llm.invoke("구체적인 시간 장소가 적혀서, 추후 메모해야 할 것 같은 회의록을 500자 이내로 생성해주세요.").content
print(example)

회의록
일시: 2026-07-15 14:00~15:30
장소: 본사 3층 대회의실
참석: 김민수, 이지영, 박준호, 최은서
안건: 신제품 출시 일정, 마케팅 예산, QA 일정
결정: 출시일 2026-09-10 확정, 마케팅 예산 1억 증액
액션: 김민수—생산 리드타임 확인(7/22), 이지영—광고안 초안 제출(7/20), 박준호—QA 일정 확정(7/25)


In [22]:
prompt = ChatPromptTemplate(
    [
        ('system', '당신은 주어진 정보에서 시간과 장소를 추출해야 합니다. 형식은 json으로 출력하세요.'),
        ('human','{article}')
    ]
)

prompt.format_messages(article=example)

response = llm.invoke(prompt.format_messages(article=example))
print(response.content)

{
  "일시": {
    "시작": "2026-07-15T14:00:00",
    "종료": "2026-07-15T15:30:00"
  },
  "장소": "본사 3층 대회의실"
}


## Few-Shot Prompting
Few-Shot Prompt Template을 이용하면, 예시를 쉽게 찾을 수 있습니다.

In [23]:
questions = [
    '당신은 어떤 모델입니까?',
    '인공지능 배워야 하나요?',
    '파이썬은 뭐가 좋나요?'
]

prompt = ChatPromptTemplate(['다음 질문에 대해서 10자 이내로 아주 간결하고 건조하게 설명해주세요. \n{question}'])

llm.invoke(prompt.format(question=questions[2]))

AIMessage(content='문법간결', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 781, 'prompt_tokens': 39, 'total_tokens': 820, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 768, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DzbZBsERTxs5AH7RuOLn4ULED6eSc', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f4557-6d3d-7290-9309-abd01e7b0382-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 39, 'output_tokens': 781, 'total_tokens': 820, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 768}})

In [25]:
prompts = []
for i, question in enumerate(questions) :
    prompts.append(prompt.format(question=questions[i]))
prompts
llm.batch(prompts)

[AIMessage(content='GPT-4', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 588, 'prompt_tokens': 37, 'total_tokens': 625, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 576, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DzbhROSII0WpVmtDpPcYD5TL9B2hR', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f455f-3874-7862-8203-8bfbc266152b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 37, 'output_tokens': 588, 'total_tokens': 625, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 576}}),
 AIMessage(content='필요에 따라', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 525, '

In [26]:
examples =[

    {'question': '당신은 어떤 모델입니까?', 'answer': '언어모델'},
    {'question': '인공지능 배워야 하나요?', 'answer': '권장됨'},
    {'question': '파이썬은 뭐가 좋나요?', 'answer': '문법간결'},
]

example_prompt = PromptTemplate(template='질문: {question}\n답변: {answer}')

fewshotprompt = FewShotPromptTemplate(
    examples = examples,
    example_prompt = example_prompt,

    prefix ='주어진 예시 형식을 참고하여 답변하세요.',
    suffix = '질문: {question} \n답변:'
)

test = fewshotprompt.format(question = '원래 말투가 그래?')
print(test)

print(llm.invoke(test).content)

주어진 예시 형식을 참고하여 답변하세요.

질문: 당신은 어떤 모델입니까?
답변: 언어모델

질문: 인공지능 배워야 하나요?
답변: 권장됨

질문: 파이썬은 뭐가 좋나요?
답변: 문법간결

질문: 원래 말투가 그래? 
답변:
질문: 원래 말투가 그래? 
답변: 상황에 따라


Chat Message에 Few Shot을 적용하는 경우, 아래와 같이 나타납니다.

In [ ]:
from langchain_core.prompts.few_shot import FewShotChatMessagePromptTemplate

example_prompt = ChatPromptTemplate(
        [
        ("human", "{question}"),
        ("ai", "{answer}"),
    ]
)
few_shot_prompt = FewShotChatMessagePromptTemplate(

    example_prompt=example_prompt,
    examples=examples,
)

few_shot_prompt.format_messages()


[HumanMessage(content='당신은 어떤 모델입니까?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='언어모델', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='인공지능 배워야 하나요?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='권장됨', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='파이썬은 뭐가 좋나요?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='문법간결', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [ ]:
final_prompt = ChatPromptTemplate(
    [
        ("system", "다음 예시를 참고하여 같은 형식으로 답변하세요."),
        few_shot_prompt,
        ("human", "{input}"),
    ]
)
final_prompt.format_messages(input="휴가 신청은 어떻게 해야 하나요?")
response = llm.invoke(final_prompt.format_messages(input="휴가 신청은 어떻게 해야 하나요?"))
print(response.content)

상사승인후인사팀제출


In [ ]:
chain = final_prompt|llm
response = chain.invoke({'input' : "휴가 신청은 어떻게 해야 하나요?"})
print(response.content)

상사승인후인사신청


### 멀티모달 프롬프트 전달하기

멀티모달 모델의 입력은 OpenAI API와 동일합니다.

In [ ]:
import base64
import httpx

# Test 이미지 URL
image_url = '''https://i.pinimg.com/originals/d9/8f/ea/d98fea1d8e5d2ef7f844aa8c0a501dd7.jpg'''
response = httpx.get(image_url)

# dog.jpg에 저장
with open('dog1.jpg', 'wb') as file:
    file.write(response.content)

In [ ]:
prompt = ChatPromptTemplate(
    [
        ('human', [
            {'type':'image', 'image_url':'{image_url}'},
            {'type':'text', 'text':'이 그림을 자세히 묘사하세요'}

                   ])
    ]
)
llm.invoke(prompt.format_messages(image_url = image_url))



AIMessage(content='사진에는 작고 단단한 체구의 개 한 마리가 풀밭 위에 서 있는 모습이 담겨 있습니다. 전체적으로 연한 황갈색(베이지·크림)에 가까운 털빛을 가지고 있고, 얼굴과 귀 주변은 짙은 갈색 또는 검은색으로 대비되어 있어 전형적인 퍼그(pug) 품종의 외형을 보입니다. 얼굴은 납작하고 주름이 잡혀 있으며, 큰 둥근 눈과 짧은 주둥이, 검은 코가 도드라져 표정이 강하게 보입니다. 귀는 접혀 있고 머리를 약간 돌려 카메라 쪽을 바라보는 듯해 호기심이나 경계심이 느껴집니다.\n\n몸집은 짧고 통통하며 등과 옆구리 라인이 매끄럽고, 짧은 다리가 땅을 단단히 딛고 있는 자세입니다. 꼬리는 등 위로 말려 올라가 있어 퍼그 특유의 말린 꼬리 모양을 보여 줍니다. 털은 짧고 빛을 받아 약간 광택이 나며, 목 주변이나 가슴 부분의 근육 윤곽이 보입니다.\n\n배경은 잔디가 깔린 야외 공간으로 녹색의 풀밭이 넓게 펼쳐져 있고, 사진 전체에 햇빛이 고르게 비쳐 밝고 선명한 분위기입니다. 전경의 개에 초점이 맞춰져 있어 배경은 약간 덜 선명해 보이며, 사진의 구도는 개의 옆모습과 정면이 섞인 3/4 측면으로 안정감 있게 잡혀 있습니다. 전체적으로 건강해 보이고 경계심 있으면서도 호기심 많은 표정의 작은 개를 촬영한 사진입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 794, 'prompt_tokens': 122, 'total_tokens': 916, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 384, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_

In [ ]:
# 로컬 이미지 전달
with open('./dog1.jpg', 'rb') as image_file:
    image_data = base64.b64encode(image_file.read()).decode('utf-8')


image_prompt = ChatPromptTemplate([
    ('user',[
                {"type": "image_url",
                    "image_url": {"url": "data:image/jpeg;base64,{image_data}"}
                },
                {"type": "text", "text": "{question}"},
             ]
     )])

X = llm.invoke(image_prompt.format_messages(
    question='이 사진을 자세히 묘사해주세요. 강아지의 품종은 설명하지 마세요.',
    image_data=image_data))

print(X.content)

사진에는 작고 동그란 흰 강아지가 중앙에 놓여 있습니다. 전체적으로 털이 아주 풍성하고 부드러워 보이며, 마치 작은 솜뭉치나 구름처럼 둥글게 부풀어 있습니다. 동그란 큰 눈은 검고 반사광이 비쳐 반짝이며, 매우 또렷하고 호기심 어린 표정을 줍니다. 코와 입은 아주 작고 검은 점처럼 보이며 얼굴은 전체적으로 귀엽고 앙증맞은 인상을 줍니다. 네 다리는 짧고 발은 털에 거의 가려져 있어 자세가 아담해 보입니다. 강아지는 벤치의 나무 널빤지 위에 서 있는데, 벤치의 나무색은 약간 바랜 갈색이고 옆에는 검은색의 금속 팔걸이가 보입니다. 배경은 약간 흐릿해 강아지에 시선이 집중되고, 자연광 아래에서 털의 결과 그림자가 은은하게 드러납니다. 전체적으로 포근하고 사랑스러운 순간을 포착한 사진입니다.
